<hr style="border:none;height:6px;background:#fff;margin:1em 0;">


<div style="text-align: center;">
  <h1>Dividend Taxation and Top-Income Inequality</h1>
  <h3>HEC Liege</h3>
  <h4><em>Lucas Dubois</em></h4>
</div>

<hr style="border:none;height:6px;background:#fff;margin:1em 0;">


<hr style="border:none;height:4px;background:#fff;margin:1em 0;">


In [1]:
import numpy as np
import pandas as pd
import cvxpy as cp
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
path = '/Users/lucasdubois/Desktop/MASTERDATA/' 
df = pd.read_csv(path + "MASTER_FINAL.csv")
df = df.sort_values(["Country Name", "Year"])
df2 =pd.read_csv(path + "MASTER_FINAL_ADJ.csv")
df2 = df2.sort_values(["Country Name", "Year"])

In [2]:
inequality_variables = [
    "top10_inc",
    "top1_inc",
    "top10_w",
    "top1_w",
    "gini_disp",
    "gini_mkt",
    "gini_disp_se",
    "gini_mkt_se"
]

interaction_variables = [
    "log_rgdpe",
    "log_pop",
    "log_emp",
    "log_cn",
    "rgdpe_sq",
    "log_rgdpe_sq",
    "hc_ctfp",
    "labsh_irr",
    "cn_ctfp",
    "xr_irr",
    "csh_g_hc",
    "csh_c_hc",
    "rgdpe_ctfp",
    "pop_rgdpe",
    "delta_cn"
]

id_variables = ["Country Name", "Year"]

X = [
    col for col in df.columns
    if col not in inequality_variables + interaction_variables + id_variables
]

X_numeric = df[X].select_dtypes(include=["number"]).columns.tolist()

<hr style="border:none;height:4px;background:#fff;margin:1em 0;">


<div style="text-align: align;">
  <h2> <small>4</small>&nbsp;&nbsp;&nbsp;&nbsp; ASCM:</h2>
</div>

<hr style="border:none; border-top:2px dashed #fff; margin:1em 0;">

<div style="text-align: align;">
  <h3> <small>3.1</small>&nbsp;&nbsp;&nbsp;&nbsp;Preparing Brazil:</h3>
</div>

In [3]:
def scm_ascm(
    df,
    country,
    treat_year,
    covariates,
    ridge_lambda=1.0,
    outcome_weight=1000.0,
    covariate_weight=1.0,
):

    # -----------------------
    # 1) PREP DATA
    # -----------------------
    df = df.sort_values(["Country Name", "Year"]).copy()

    needed_cols = ["Country Name", "Year", "top1_inc"] + covariates_ascm
    df = df[needed_cols].copy()

    years = sorted(df["Year"].unique())
    pre_years = [y for y in years if y < treat_year]
    post_years = [y for y in years if y >= treat_year]

    # Outcome matrix
    Y = df.pivot(index="Year", columns="Country Name", values="top1_inc").sort_index()

    # Covariate matrices
    cov_mats = {
        cov: df.pivot(index="Year", columns="Country Name", values=cov).sort_index()
        for cov in covariates
    }

    # Keep only countries with complete pre-treatment outcome + covariate data
    valid = Y.loc[pre_years].notna().all(axis=0)
    for cov in covariates:
        valid = valid & cov_mats[cov].loc[pre_years].notna().all(axis=0)

    Y = Y.loc[:, valid]
    cov_mats = {cov: mat.loc[:, valid] for cov, mat in cov_mats.items()}

    if country not in Y.columns:
        raise ValueError(f"{country} not in columns after cleaning.")

    donors = [c for c in Y.columns if c != country]

    Y1 = Y[country]
    Y0 = Y[donors]

    # -----------------------
    # 2) BUILD MATCHING BLOCKS
    # -----------------------
    # Outcome path in pre-treatment period
    X0_pre_outcome = Y0.loc[pre_years].values          # T0 x J
    y1_pre_outcome = Y1.loc[pre_years].values          # T0

    # Covariate block: use pre-treatment means
    treated_cov_list = []
    donor_cov_list = []

    for cov in covariates:
        treated_mean = cov_mats[cov].loc[pre_years, country].mean()
        donor_means = cov_mats[cov].loc[pre_years, donors].mean(axis=0).values

        treated_cov_list.append(treated_mean)
        donor_cov_list.append(donor_means)

    if len(covariates) > 0:
        y1_cov = np.array(treated_cov_list)            # K
        X0_cov = np.vstack(donor_cov_list)             # K x J
    else:
        y1_cov = np.array([])
        X0_cov = np.empty((0, len(donors)))

    # Optional weighting: keeps outcome path more important if desired
    y1_pre_outcome = outcome_weight * y1_pre_outcome
    X0_pre_outcome = outcome_weight * X0_pre_outcome

    y1_cov = covariate_weight * y1_cov
    X0_cov = covariate_weight * X0_cov

    # Stack into one matching problem
    Z1 = np.concatenate([y1_pre_outcome, y1_cov])     # (T0 + K,)
    Z0 = np.vstack([X0_pre_outcome, X0_cov])          # (T0 + K) x J

    # -----------------------
    # 3) STANDARDIZE MATCHING ROWS
    #    critical when adding covariates
    # -----------------------
    row_means = Z0.mean(axis=1)
    row_stds = Z0.std(axis=1, ddof=0)
    row_stds[row_stds == 0] = 1.0

    Z0_scaled = (Z0 - row_means[:, None]) / row_stds[:, None]
    Z1_scaled = (Z1 - row_means) / row_stds

    Tmatch, J = Z0_scaled.shape

    # -----------------------
    # 4) SCM WITH COVARIATES
    # -----------------------
    w = cp.Variable(J)

    objective = cp.Minimize(cp.sum_squares(Z1_scaled - Z0_scaled @ w))
    constraints = [w >= 0, cp.sum(w) == 1]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.OSQP)

    if w.value is None:
        raise RuntimeError("SCM optimization failed. Try solver=cp.SCS.")

    w_scm = np.array(w.value).ravel()
    weights_scm = pd.Series(w_scm, index=donors).sort_values(ascending=False)

    synthetic_scm = pd.Series(Y0.values @ w_scm, index=Y.index, name="Synthetic_SCM")

    # -----------------------
    # 5) RIDGE-ASCM
    # -----------------------
    pre_gap_match = Z1_scaled - Z0_scaled @ w_scm

    A = Z0_scaled @ Z0_scaled.T + ridge_lambda * np.eye(Tmatch)
    bias = Z0_scaled.T @ np.linalg.solve(A, pre_gap_match)

    w_ascm = w_scm + bias
    weights_ascm = pd.Series(w_ascm, index=donors).sort_values(ascending=False)

    synthetic_ascm = pd.Series(Y0.values @ w_ascm, index=Y.index, name="Synthetic_ASCM")

    # -----------------------
    # 6) EFFECTS + DIAGNOSTICS
    # -----------------------
    effect_scm = Y1 - synthetic_scm
    effect_ascm = Y1 - synthetic_ascm

    pre_rmse_scm = np.sqrt(np.mean((Y1.loc[pre_years] - synthetic_scm.loc[pre_years]) ** 2))
    pre_rmse_ascm = np.sqrt(np.mean((Y1.loc[pre_years] - synthetic_ascm.loc[pre_years]) ** 2))

    pre_sse_scm = np.sum((Y1.loc[pre_years] - synthetic_scm.loc[pre_years]) ** 2)
    pre_sse_ascm = np.sum((Y1.loc[pre_years] - synthetic_ascm.loc[pre_years]) ** 2)

    print(f"Pre-treatment RMSE (SCM):  {pre_rmse_scm:.4f}")
    print(f"Pre-treatment RMSE (ASCM): {pre_rmse_ascm:.4f}")

    print("\nTop SCM donor weights:")
    print(weights_scm.head(15))

    print("\nTop ASCM donor weights:")
    print(weights_ascm.head(15))

    print("\nMost negative ASCM weights:")
    print(weights_ascm.sort_values().head(10))

    return {
        "country": country,
        "treat_year": treat_year,
        "covariates": covariates,
        "ridge_lambda": ridge_lambda,
        "outcome_weight": outcome_weight,
        "covariate_weight": covariate_weight,
        "years": years,
        "pre_years": pre_years,
        "post_years": post_years,
        "Y": Y,
        "Y1": Y1,
        "Y0": Y0,
        "weights_scm": weights_scm,
        "weights_ascm": weights_ascm,
        "synthetic_scm": synthetic_scm,
        "synthetic_ascm": synthetic_ascm,
        "effect_scm": effect_scm,
        "effect_ascm": effect_ascm,
        "pre_rmse_scm": pre_rmse_scm,
        "pre_rmse_ascm": pre_rmse_ascm,
        "pre_sse_scm": pre_sse_scm,
        "pre_sse_ascm": pre_sse_ascm,
        "Z0_scaled": Z0_scaled,
        "Z1_scaled": Z1_scaled,
    }

In [4]:
covariates = [    "log_rgdpe",
    "log_pop",
    "log_emp",
    "log_cn",]

brazil_ascm = scm_ascm(
    df=df2,
    country="Brazil",
    treat_year=1996,
    covariates=covariates,
    ridge_lambda=1.0
)

Pre-treatment RMSE (SCM):  0.0183
Pre-treatment RMSE (ASCM): 0.0107

Top SCM donor weights:
Thailand          9.286324e-01
United States     7.136760e-02
Iran             -5.875444e-15
India            -6.677483e-15
Madagascar       -8.434721e-15
Philippines      -1.093899e-14
Japan            -1.101767e-14
Botswana         -1.263798e-14
Malaysia         -1.916598e-14
Germany          -2.096262e-14
Pakistan         -2.326040e-14
France           -2.954473e-14
United Kingdom   -3.111309e-14
Israel           -3.138776e-14
Italy            -3.187411e-14
dtype: float64

Top ASCM donor weights:
Thailand         0.947570
Japan            0.114329
Madagascar       0.113827
Norway           0.097065
South Korea      0.081910
Iran             0.081107
Singapore        0.056197
Belgium          0.051699
Denmark          0.051543
Switzerland      0.043437
Germany          0.040837
Ghana            0.037391
United States    0.037063
Egypt            0.034417
Nigeria          0.025188
dtype: float6

In [5]:
covariates = [    "log_rgdpe",
    "log_pop",
    "log_emp",
    "log_cn",]

usa_ascm = scm_ascm(
    df=df,
    country="United States",
    treat_year=2003,
    covariates=covariates,
    ridge_lambda=1.0
)

Pre-treatment RMSE (SCM):  0.0118
Pre-treatment RMSE (ASCM): 0.0055

Top SCM donor weights:
India             4.325873e-01
Japan             3.108904e-01
Brazil            1.544311e-01
Philippines       1.020912e-01
Thailand         -9.017934e-17
Iran             -2.015634e-16
Germany          -2.636747e-16
United Kingdom   -4.290212e-16
France           -4.569300e-16
Botswana         -5.038643e-16
Pakistan         -5.383626e-16
Italy            -5.539262e-16
Malaysia         -5.893814e-16
Egypt            -7.411518e-16
Spain            -7.507869e-16
dtype: float64

Top ASCM donor weights:
India          0.345588
Japan          0.290195
Brazil         0.127755
Philippines    0.117987
Canada         0.102561
Israel         0.100614
Australia      0.098219
Germany        0.081586
Luxembourg     0.077887
Italy          0.076998
France         0.072646
Hong Kong      0.062639
Thailand       0.062146
New Zealand    0.061340
Botswana       0.044831
dtype: float64

Most negative ASCM weights:

In [ ]:
def scm_ascm(
    df,
    country,
    treat_year,
    covariates
):

    # -----------------------
    # 1) PREP DATA
    # -----------------------
    df = df.sort_values(["Country Name", "Year"]).copy()
    needed_cols = ["Country Name", "Year", "top1_inc"] + covariates_ascm
    df = df[needed_cols].copy()

    years = sorted(df["Year"].unique())
    pre_years = [y for y in years if y < treat_year]
    post_years = [y for y in years if y >= treat_year]

    # Outcome matrix
    Y = df.pivot(index="Year", columns="Country Name", values="top1_inc").sort_index()

    # Covariate matrices
    X = {
        cov: df.pivot(index="Year", columns="Country Name", values=cov).sort_index()
        for cov in covariates
    }

    # Keep only countries with complete pre-treatment outcome + covariate data
    valid = Y.loc[pre_years].notna().all(axis=0)
    for cov in covariates:
        valid = valid & X[cov].loc[pre_years].notna().all(axis=0)

    Y = Y.loc[:, valid]
    X = {cov: mat.loc[:, valid] for cov, mat in X.items()}

    if country not in Y.columns:
        raise ValueError(f"{country} not in columns after cleaning.")

    donors = [c for c in Y.columns if c != country]

    Y1 = Y[country]
    Y0 = Y[donors]
    X1 = X[country]
    X0 = X[donors]

    # -----------------------
    # 2) BUILD MATCHING BLOCKS
    # -----------------------
    # Outcome path in pre-treatment period
    Y_train = Y.loc[pre_years].values  
    X_train = X.loc[pre_years].values
    Y0_train = Y0.loc[pre_years].values          
    Y1_train = Y1.loc[pre_years].values
    X0_train = X0.loc[pre_years].values          
    X1_train = X1.loc[pre_years].values 

    Y_test = Y.loc[post_years].values  
    X_test = X.loc[post_years].values
    Y0_test = Y0.loc[post_years].values          
    Y1_test = Y1.loc[post_years].values
    X0_test = X0.loc[post_years].values          
    X1_test = X1.loc[post_years].values


    # -----------------------
    # 4) SCM:
    # -----------------------
    w = cp.Variable(J)

    objective = cp.Minimize(cp.sum_squares(Y1_train - Y0_train @ w))
    constraints = [w >= 0, cp.sum(w) == 1]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.OSQP)

    if w.value is None:
        raise RuntimeError("SCM optimization failed. Try solver=cp.SCS.")

    w_scm = np.array(w.value).ravel()
    weights_scm = pd.Series(w_scm, index=donors).sort_values(ascending=False)

    synthetic_scm = pd.Series(Y0.values @ w_scm, index=Y.index, name="Synthetic_SCM")

    # -----------------------
    # 5) RIDGE-Model
    # -----------------------
    
    scale = MinMaxScaler()
    X_train_transformed = scale.fit_transform(X_train)
    X_test_transformed = scale.transform(X_test)
    X0_test_transformed = scale.transform(X0_test)

    ridge = Ridge(alpha=0.5)
    ridge.fit(X_train_transformed, Y_train)
    m1t = ridge.predict(X0_test_transformed)
    
    
    
    pre_gap_match = Z1_scaled - Z0_scaled @ w_scm
    

    A = Z0_scaled @ Z0_scaled.T + ridge_lambda * np.eye(Tmatch)
    bias = Z0_scaled.T @ np.linalg.solve(A, pre_gap_match)

    w_ascm = w_scm + bias
    weights_ascm = pd.Series(w_ascm, index=donors).sort_values(ascending=False)

    synthetic_ascm = pd.Series(Y0.values @ w_ascm, index=Y.index, name="Synthetic_ASCM")

    # -----------------------
    # 6) EFFECTS + DIAGNOSTICS
    # -----------------------
    effect_scm = Y1 - synthetic_scm
    effect_ascm = Y1 - synthetic_ascm

    pre_rmse_scm = np.sqrt(np.mean((Y1.loc[pre_years] - synthetic_scm.loc[pre_years]) ** 2))
    pre_rmse_ascm = np.sqrt(np.mean((Y1.loc[pre_years] - synthetic_ascm.loc[pre_years]) ** 2))

    pre_sse_scm = np.sum((Y1.loc[pre_years] - synthetic_scm.loc[pre_years]) ** 2)
    pre_sse_ascm = np.sum((Y1.loc[pre_years] - synthetic_ascm.loc[pre_years]) ** 2)

    print(f"Pre-treatment RMSE (SCM):  {pre_rmse_scm:.4f}")
    print(f"Pre-treatment RMSE (ASCM): {pre_rmse_ascm:.4f}")

    print("\nTop SCM donor weights:")
    print(weights_scm.head(15))

    print("\nTop ASCM donor weights:")
    print(weights_ascm.head(15))

    print("\nMost negative ASCM weights:")
    print(weights_ascm.sort_values().head(10))

    return {
        "country": country,
        "treat_year": treat_year,
        "covariates": covariates,
        "ridge_lambda": ridge_lambda,
        "outcome_weight": outcome_weight,
        "covariate_weight": covariate_weight,
        "years": years,
        "pre_years": pre_years,
        "post_years": post_years,
        "Y": Y,
        "Y1": Y1,
        "Y0": Y0,
        "weights_scm": weights_scm,
        "weights_ascm": weights_ascm,
        "synthetic_scm": synthetic_scm,
        "synthetic_ascm": synthetic_ascm,
        "effect_scm": effect_scm,
        "effect_ascm": effect_ascm,
        "pre_rmse_scm": pre_rmse_scm,
        "pre_rmse_ascm": pre_rmse_ascm,
        "pre_sse_scm": pre_sse_scm,
        "pre_sse_ascm": pre_sse_ascm,
        "Z0_scaled": Z0_scaled,
        "Z1_scaled": Z1_scaled,
    }